# mySports Video Creator 🏒

Dieses Notebook schneidet die mySports-Reels im freigegebenen Design (9:16, Figtree Bold, Sunrise Red).
Texte, Untertitel und Specs kommen vom **Gemini-Gem «mySports Video Creator»**, hier wird nur gerendert.

**So gehts:** Zellen von oben nach unten mit ▶ ausführen. Schritt 1 immer zuerst.
Die Videos liegen im Google Drive im Projektordner, nie auf GitHub.

## 1 · Einrichten (jedes Mal zuerst)

In [ ]:
#@title Projektordner auf Google Drive { display-mode: "form" }
ORDNER = "mySports - SOM Organic - Video Creator"  #@param {type:"string"}
REPO = "https://github.com/dan-b-h/mysports-video-creator.git"

from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, glob

class Hinweis(Exception):
    """Bricht eine Zelle mit einer kurzen Meldung ab, ohne technischen Fehlertext."""
    def _render_traceback_(self):
        return ['', '⚠️  ' + str(self), '']
# Schnelle Suche: nur direkt in «Meine Ablage», eine Ebene darunter und in den geteilten Ablagen.
# Tiefer wird bewusst nicht gesucht, das dauert über Drive viele Minuten. Liegt der Ordner tiefer,
# einmalig eine Verknüpfung in «Meine Ablage» anlegen (siehe README, «Einmalig einrichten»).
e = glob.escape(ORDNER)
muster = ['/content/drive/MyDrive/' + e, '/content/drive/MyDrive/*/' + e,
          '/content/drive/Shareddrives/*/' + e, '/content/drive/.shortcut-targets-by-id/*/' + e]
PROJEKT = None
for m in muster:
    treffer = [p for p in glob.glob(m) if os.path.isdir(p)]
    if treffer:
        PROJEKT = treffer[0]; break
if not PROJEKT:
    raise Hinweis(f"Ordner «{ORDNER}» nicht gefunden.\n"
                  "    Lege im Google Drive einmalig eine Verknüpfung an: Ordner öffnen, oben auf den Ordnernamen klicken,\n"
                  "    «Organisieren» → «Verknüpfung hinzufügen» → «Meine Ablage». Danach diese Zelle neu starten.")
print('Projektordner:', PROJEKT)
if os.path.isdir('/content/mvc'):
    subprocess.run(['git', '-C', '/content/mvc', 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', REPO, '/content/mvc'], check=True)
# Brand-Dateien (Schriften, Logo) liegen auf Drive und werden für die Skripte verlinkt
if not os.path.exists('/content/mvc/03_Brand'):
    os.symlink(os.path.join(PROJEKT, '03_Brand'), '/content/mvc/03_Brand')
os.chdir(PROJEKT)
V = '/content/mvc/vorlage'
print('Bereit. Arbeitsordner:', os.getcwd())
print('Originalvideos:', len(os.listdir('Originalvideos')), 'Dateien')

## 2 · Video sichten (optional)
Erstellt Kontaktbögen mit Zeitstempeln, damit du Tore und Szenen schnell findest.
Die Bilder landen in `04_Arbeitsdateien/<Datum>/check/`.

In [ ]:
#@title Kontaktbogen { display-mode: "form" }
VIDEO = "Originalvideos/261004_HCA_HCL_DE.mp4"  #@param {type:"string"}
DATUM = "261004"  #@param {type:"string"}
import glob
from IPython.display import Image, display
out = f"04_Arbeitsdateien/{DATUM}/check"; os.makedirs(out, exist_ok=True)
name = os.path.splitext(os.path.basename(VIDEO))[0]
font = os.path.join(PROJEKT, '03_Brand', 'Figtree-Bold.ttf')
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-skip_frame', 'nokey', '-i', VIDEO, '-vf',
    f"scale=320:-1,drawtext=fontfile='{font}':text='%{{pts\\:hms}}':x=4:y=4:fontsize=20:fontcolor=yellow:box=1:boxcolor=black,tile=6x6",
    '-vsync', 'vfr', f'{out}/{name}_%02d.png'], check=True)
for p in sorted(glob.glob(f'{out}/{name}_*.png')): display(Image(p, width=1100))

## 3 · Interview transkribieren (nur für Interview- und Kombi-Reels)
Erstellt ein Transkript mit Zeitangaben. Das Transkript gibst du dem Gem; er liefert dir die bereinigten Untertitel zurück.
Beim ersten Mal werden die Spracherkennungs-Modelle geladen (rund 1 GB, ca. 3–5 Minuten, Zelle bitte nicht abbrechen), danach liegen sie im Drive.

In [ ]:
#@title Transkript { display-mode: "form" }
VIDEO = "Originalvideos/261003_SCRJ_SCB_INTERVIEWS_DE.mp4"  #@param {type:"string"}
DATUM = "261003"  #@param {type:"string"}
SPRACHE = "de"  #@param ["de", "fr", "en", "it"]
subprocess.run('pip -q install sherpa-onnx', shell=True, check=True)
# Modelle: einmal von GitHub laden, dann im Drive (_modelle) zwischenspeichern.
# Geladen und entpackt wird zuerst lokal in Colab; ins Drive kommt nur ein vollständiger Satz (Marker .fertig).
import shutil
MOD = os.path.join(PROJEKT, '_modelle')
LOKAL = '/content/modelle'
NOETIG = ['sherpa-onnx-whisper-turbo/turbo-encoder.int8.onnx', 'sherpa-onnx-whisper-turbo/turbo-decoder.int8.onnx',
          'sherpa-onnx-whisper-turbo/turbo-tokens.txt', 'silero_vad.onnx']
def komplett(d):
    return os.path.exists(f'{d}/.fertig') and all(os.path.exists(f'{d}/{f}') for f in NOETIG)
if not komplett(LOKAL):
    if komplett(MOD):
        print('Lade Spracherkennung aus dem Drive …')
        shutil.copytree(MOD, LOKAL, dirs_exist_ok=True)
    else:
        print('Lade Spracherkennung von GitHub (nur beim ersten Mal, rund 1 GB, ca. 3–5 Minuten) …')
        shutil.rmtree(LOKAL, ignore_errors=True); os.makedirs(LOKAL)
        rel = 'https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models'
        subprocess.run(['bash', '-c', f'set -o pipefail; cd {LOKAL} && curl -fsSL {rel}/sherpa-onnx-whisper-turbo.tar.bz2 | tar xj '
                        f'&& curl -fsSL -o silero_vad.onnx {rel}/silero_vad.onnx'], check=True)
        fehlt = [f for f in NOETIG if not os.path.exists(f'{LOKAL}/{f}')]
        if fehlt:
            raise Hinweis('Download der Spracherkennung unvollständig, es fehlt: ' + ', '.join(fehlt) + '. Zelle einfach nochmals starten.')
        open(f'{LOKAL}/.fertig', 'w').close()
        print('Speichere Spracherkennung im Drive (_modelle) für das nächste Mal …')
        shutil.rmtree(MOD, ignore_errors=True)        # alte, halbe Downloads entfernen
        shutil.copytree(LOKAL, MOD, ignore=shutil.ignore_patterns('.fertig', 'test_wavs'))
        open(f'{MOD}/.fertig', 'w').close()           # Marker erst ganz am Schluss
os.makedirs(f'04_Arbeitsdateien/{DATUM}/audio', exist_ok=True)
WAV = f"04_Arbeitsdateien/{DATUM}/audio/{os.path.splitext(os.path.basename(VIDEO))[0]}.wav"
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', VIDEO, '-vn', '-ac', '1', '-ar', '16000', WAV], check=True)
env = dict(os.environ, MODELLE=LOKAL, SPRACHE=SPRACHE)
r = subprocess.run(['python3', f'{V}/audio_tools.py', 'transcribe', WAV], capture_output=True, text=True, env=env)
txt = '\n'.join(l for l in r.stdout.splitlines() if not l.startswith('/project'))
open(WAV.replace('.wav', '_Transkript.txt'), 'w').write(txt)
if r.returncode != 0:
    print(r.stderr[-1500:])
    raise Hinweis('Transkription fehlgeschlagen (Meldung siehe oben).')
print(txt)
print('\nSprechpausen (für saubere Schnitte):')
print(subprocess.run(['python3', f'{V}/audio_tools.py', 'pauses', WAV], capture_output=True, text=True, env=env).stdout)

## 4 · Spec speichern

**① Einfügen:** In der ersten Zelle die Zeile `HIER_SPEC_EINFÜGEN` markieren und durch die Spec (JSON) vom Gem ersetzen. Die Zeilen `SPEC = r"""` und `"""` bleiben stehen. Am einfachsten: alles zwischen diesen beiden Zeilen markieren und einfügen.

**② Speichern:** Danach die zweite Zelle ausführen. Sie prüft die Spec und meldet klar, wenn etwas fehlt.

Bei Interview- und Kombi-Reels werden die Wortzeiten der Untertitel automatisch berechnet.

In [ ]:
# ① Spec vom Gem hier einfügen: die Zeile HIER_SPEC_EINFÜGEN ersetzen (die Zeilen mit """ bleiben stehen)
SPEC = r"""
HIER_SPEC_EINFÜGEN
"""


In [ ]:
#@title ② Spec prüfen und speichern { display-mode: "form" }
DATUM = "261004"  #@param {type:"string"}
NAME = "hca_split_v1"  #@param {type:"string"}
import json
try:
    SPEC
except NameError:
    raise Hinweis('Zuerst die Zelle ① oben ausführen (Spec einfügen).')
roh = SPEC.strip()
if not roh or 'HIER_SPEC_EINF' in roh or 'JSON vom Gem' in roh:
    raise Hinweis('In Zelle ① steht noch keine Spec. Spec vom Gem einfügen, Zelle ① und danach diese Zelle ausführen.')
if '{' in roh and '}' in roh:                 # Reste wie ```json am Anfang oder Ende ignorieren
    roh = roh[roh.index('{'):roh.rindex('}') + 1]
try:
    spec = json.loads(roh)
except json.JSONDecodeError as fehler:
    zeile = roh.splitlines()[fehler.lineno - 1] if fehler.lineno <= len(roh.splitlines()) else ''
    raise Hinweis(f'Die Spec ist kein gültiges JSON (Zeile {fehler.lineno}, Zeichen {fehler.colno}).\n'
                  f'    Dort steht: {zeile[max(0, fehler.colno - 40):fehler.colno + 40]}\n'
                  '    Spec nochmals vollständig vom Gem kopieren und in Zelle ① einfügen.')
# Zeiten dürfen als "M:SS.s" kommen (z. B. "1:39.5"), so wie der Gem sie im Video sieht. Hier in Sekunden umrechnen.
ZEITFELDER = {'start', 'end', 'goal_at', 'src_s', 'src_e', 'bottom_still_at'}
def sek(v):
    if isinstance(v, str) and ':' in v:
        m, s_ = v.strip().split(':')
        return round(int(m) * 60 + float(s_.replace(',', '.')), 2)
    if isinstance(v, str):
        return float(v.replace(',', '.'))
    return v
def umrechnen(x, schluessel=None):
    if isinstance(x, dict):
        return {k: (sek(v) if k in ZEITFELDER else umrechnen(v, k)) for k, v in x.items()}
    if isinstance(x, list):
        if schluessel == 'segments' and x and isinstance(x[0], list):
            return [[sek(a) for a in teil] for teil in x]
        return [umrechnen(v, schluessel) for v in x]
    return x
try:
    spec = umrechnen(spec)
except ValueError as fehler:
    raise Hinweis(f'Eine Zeitangabe in der Spec ist nicht lesbar ({fehler}). Format: "1:39.5" oder 99.5.')

quellen = [q for q in (spec.get('source'), spec.get('goals', {}).get('source'), spec.get('interview', {}).get('source')) if q]
if not quellen:
    raise Hinweis('In der Spec fehlt das Quellvideo ("source"). Ist es wirklich die Spec vom Gem?')
for q in quellen:
    if not os.path.exists(q):
        raise Hinweis(f'Video «{q}» nicht gefunden. Liegt es im Ordner Originalvideos und stimmt der Dateiname?')
# Plausibilität: Tor innerhalb der Szene, Szene innerhalb des Videos
def laenge(q):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', q], capture_output=True, text=True)
    return float(r.stdout.strip() or 0)
for teil in (spec, spec.get('goals', {}), spec.get('interview', {})):
    if not teil.get('source'):
        continue
    dauer = laenge(teil['source'])
    for seg in teil.get('segments', []):
        a, b = (seg['start'], seg['end']) if isinstance(seg, dict) else seg
        if b > dauer:
            raise Hinweis(f'Szene {a}–{b} s liegt ausserhalb des Videos (Länge {dauer:.1f} s). Zeiten prüfen; Gem nach "M:SS" fragen.')
        if isinstance(seg, dict) and 'goal_at' in seg and not (a <= seg['goal_at'] <= b):
            raise Hinweis(f'goal_at {seg["goal_at"]} s liegt nicht in der Szene {a}–{b} s ({seg.get("sub", "")}).')
pfad = f"04_Arbeitsdateien/{DATUM}/{NAME}.json"; os.makedirs(os.path.dirname(pfad), exist_ok=True)

def wortzeiten(teil, quelle):
    """Untertitel mit src_s/src_e -> Clip-Zeiten und Wort-für-Wort-Einblendung."""
    if not teil.get('cues') or 'word_t' in teil['cues'][0]: return teil
    wav = f"04_Arbeitsdateien/{DATUM}/audio/{os.path.splitext(os.path.basename(quelle))[0]}.wav"
    if not os.path.exists(wav):
        os.makedirs(os.path.dirname(wav), exist_ok=True)
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', quelle, '-vn', '-ac', '1', '-ar', '16000', wav], check=True)
    subprocess.run('pip -q install sherpa-onnx', shell=True, check=True)
    tmp_in, tmp_out = '/content/cues_in.json', '/content/cues_out.json'
    json.dump(teil, open(tmp_in, 'w'), ensure_ascii=False)
    r = subprocess.run(['python3', f'{V}/audio_tools.py', 'wordtimes', wav, tmp_in, tmp_out], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
    return json.load(open(tmp_out))

if 'interview' in spec:            # Kombi
    spec['interview'] = wortzeiten(spec['interview'], spec['interview']['source'])
elif 'cues' in spec:               # Interview
    spec = wortzeiten(spec, spec['source'])
json.dump(spec, open(pfad, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Gespeichert:', pfad)

## 5 · Rendern
Wähle das Format und die Spec. Das fertige Video landet in `05_Ausgabe/<Datum>/` und wird unten angezeigt.
**Rechte:** Pro Quelldatei höchstens 30 Sekunden. Längere Specs brechen mit einer Fehlermeldung ab (Ausnahme nur mit `"allow_over_30": true` und Freigabe).

In [ ]:
#@title Video rendern { display-mode: "form" }
FORMAT = "Tore-Splitscreen"  #@param ["Tore-Splitscreen", "Interview", "Kombi Tore + Interview"]
DATUM = "261004"  #@param {type:"string"}
NAME = "hca_split_v1"  #@param {type:"string"}
AUSGABE = "261004_DE_HCA_HCL_Tore_Split_v1"  #@param {type:"string"}
skript = {"Tore-Splitscreen": "render_goals_split.py", "Interview": "render_interview.py",
          "Kombi Tore + Interview": "render_combo.py"}[FORMAT]
ordner = {"Tore-Splitscreen": "Spielhighlights", "Interview": "Interview", "Kombi Tore + Interview": "Interview"}[FORMAT]
if not os.path.exists(f'04_Arbeitsdateien/{DATUM}/{NAME}.json'):
    raise Hinweis(f'Spec «04_Arbeitsdateien/{DATUM}/{NAME}.json» nicht gefunden. Zuerst Abschnitt 4 ausführen; DATUM und NAME müssen gleich sein wie dort.')
ziel = f"05_Ausgabe/{DATUM}/{ordner}/{AUSGABE}.mp4"; os.makedirs(os.path.dirname(ziel), exist_ok=True)
r = subprocess.run(['python3', f'{V}/{skript}', f'04_Arbeitsdateien/{DATUM}/{NAME}.json', ziel], capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise Hinweis('Rendern fehlgeschlagen (Meldung siehe oben).')
if r.returncode == 0:
    from IPython.display import HTML
    from base64 import b64encode
    klein = '/content/vorschau.mp4'
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', ziel, '-vf', 'scale=360:-2', '-c:v', 'libx264', '-crf', '28', '-c:a', 'aac', klein], check=True)
    display(HTML(f'<video width=360 controls src="data:video/mp4;base64,{b64encode(open(klein,"rb").read()).decode()}"></video>'))
    print('Fertig:', ziel)

## 6 · Kontrolle vor dem Versand
- Spielstand springt genau beim Tor um, Torschütze und Minute stimmen
- Gesicht bzw. Torszene ist gut im Bild, nichts Wichtiges abgeschnitten
- Untertitel: jedes Wort korrekt, keine Sätze abgehackt
- Anfang und Schluss sind sauber (kein halbes Wort)
- Nutzungsprotokoll in `02_Daten/nutzungsprotokoll.csv` nachgeführt (Quelldatei, Start, Ende, Summe)
- Freigabe der Kundin eingeholt